## **집계연산**

### **PySpark 설정**

In [2]:
import os
import sys
from pyspark import SparkConf, SparkContext

# 1. 실행 중인 파이썬 경로를 PySpark 환경변수에 직접 등록 (핵심 해결책)
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

# 2. 기존 SparkContext 종료
if 'sc' in locals():
    sc.stop()

# 3. 문제 조건에 맞춰 SparkConf 설정
conf = SparkConf() \
    .setMaster("local") \
    .setAppName("transformations_actions")

# 4. SparkContext 생성
sc = SparkContext(conf=conf)

# 5. RDD 실행 테스트
data = [1, 2, 3, 4, 5]
rdd = sc.parallelize(data)
result = rdd.map(lambda x: x * 10).collect()

print("테스트 결과:", result)

테스트 결과: [10, 20, 30, 40, 50]


In [3]:
# [+] pyspark 불러오기
from pyspark import SparkContext

sc = SparkContext.getOrCreate()

In [4]:
import os
import sys
from pyspark import SparkConf, SparkContext

# 1. 클래스에 남아있는 활성 SparkContext 강제 종료 (핵심 해결책)
if SparkContext._active_spark_context is not None:
    SparkContext._active_spark_context.stop()

# 2. 파이썬 환경변수 설정 (윈도우 통신 오류 방지)
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

# 3. 주석 요구사항 SparkConf 설정
conf = SparkConf() \
    .setMaster("local") \
    .setAppName("transformations_actions")

# 4. SparkContext 재생성
sc = SparkContext(conf=conf)

print("정상 연결 완료!")
print("App Name:", sc.appName)
print("Master:", sc.master)

정상 연결 완료!
App Name: transformations_actions
Master: local


### **reduce( ) 메서드**

In [5]:
# [+] x+y를 수행하는 reduce() 
sc.parallelize([1, 2, 3, 4, 5]).reduce(lambda x,y: x+y)

15

In [6]:
# reduce() 는 각 파티션별로 수행됨
# [+] 파티션 수: 2
rdd = sc.parallelize([1,2,3,4,5],2)
rdd.collect()

[1, 2, 3, 4, 5]

In [7]:
# [+] glom() : 파티션 별 RDD 값들을 배열에 모으는 변환 메서드
rdd.glom().collect()

[[1, 2], [3, 4, 5]]

In [8]:
# [+] 파티션 2개에 대해 x+y를 수행하는 reduce()
rdd.reduce(lambda x,y: x+y)

15

In [9]:
# [+] reduce() 주의점: 파티션 구성에 따라 결과 값이 달라질 수 있음
# 예: lambda x, y: (x * 2) + y
sc.parallelize([1,2,3,4,5]).reduce(lambda x,y: (x * 2) + y)

57

In [15]:
# 파티션 수: 2
sc.parallelize([1,2,3,4,5], 2).reduce(lambda x,y: (x * 2) + y)

33

In [16]:
sc.parallelize([1,2,3,4,5], 3).reduce(lambda x,y: (x * 2) + y)

31

### **10000 개 정수 난수의 총합 계산하기**

In [10]:
!pip install numpy


[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
# [+] numpy 라이브러리 불러오기
import numpy as np

In [12]:
# [+] [0, 100] 범위의 정수 난수 10000개 생성
x = np.random.randint(0, 100, size=10000)
x[:100]

array([99, 16, 70, 70, 60,  2, 39, 52, 18, 27, 32,  9, 28, 69, 25,  2, 92,
       30, 67, 70, 42, 17, 33, 15, 35, 14, 77,  0, 22, 72, 78, 93, 14, 76,
       94,  7, 90, 48, 38, 18, 85, 18, 91, 17, 60,  6, 93, 67, 72, 75, 93,
       86,  1,  9, 35, 13, 19, 42, 77, 32, 48, 87, 73, 91, 80, 73, 87, 78,
       52, 64, 30, 35, 19, 99, 52,  4, 57,  4, 54, 17, 40, 25, 38, 78, 72,
       96, 73, 51,  9, 36,  6, 10, 33, 98, 92, 68, 71, 11, 72, 15],
      dtype=int32)

In [13]:
# [+] 생성한 난수 배열을 RDD로 변환(파티션 수: 8개)
rdd = sc.parallelize(x, 8)

In [14]:
# 파티션 별 RDD 값 확인
partitioned_data = rdd.glom().collect()
for i, partition in enumerate(partitioned_data):
    print(f"Partition {i}: {partition}\n")

Partition 0: [np.int32(99), np.int32(16), np.int32(70), np.int32(70), np.int32(60), np.int32(2), np.int32(39), np.int32(52), np.int32(18), np.int32(27), np.int32(32), np.int32(9), np.int32(28), np.int32(69), np.int32(25), np.int32(2), np.int32(92), np.int32(30), np.int32(67), np.int32(70), np.int32(42), np.int32(17), np.int32(33), np.int32(15), np.int32(35), np.int32(14), np.int32(77), np.int32(0), np.int32(22), np.int32(72), np.int32(78), np.int32(93), np.int32(14), np.int32(76), np.int32(94), np.int32(7), np.int32(90), np.int32(48), np.int32(38), np.int32(18), np.int32(85), np.int32(18), np.int32(91), np.int32(17), np.int32(60), np.int32(6), np.int32(93), np.int32(67), np.int32(72), np.int32(75), np.int32(93), np.int32(86), np.int32(1), np.int32(9), np.int32(35), np.int32(13), np.int32(19), np.int32(42), np.int32(77), np.int32(32), np.int32(48), np.int32(87), np.int32(73), np.int32(91), np.int32(80), np.int32(73), np.int32(87), np.int32(78), np.int32(52), np.int32(64), np.int32(30), 

In [15]:
# [+] x+y 집계 연산
rdd.reduce(lambda x,y: x+y)

np.int32(492050)

### **fold( ) 메서드**
```reduce()```와 유사하며, 초기 값을 지정하여 집계를 수행

In [16]:
# [+] 초기 값 1이 주어지고, x+y를 수행하는 fold()
sc.parallelize([1, 2, 3, 4]).fold(1, lambda x, y: x + y)

12

In [17]:
# [+] 파티션 수가 2개이며, 초기 값 1이 주어지고, x+y를 수행하는 fold()
sc.parallelize([1, 2, 3, 4],2).fold(1, lambda x,y: x + y)

13

In [18]:
# [+] 초기 값이 0이 주어지고, x*y를 수행하는 fold()
sc.parallelize([1, 2, 3, 4]).fold(0, lambda x,y: x*y)

0

### **groupBy( ) 메서드**
```groupBy()```: 인자 함수를 기준으로 그룹핑을 수행


In [25]:
# [+] 홀수/짝수로 그룹핑을 수행하는 groupBy() 코드
res = sc.parallelize([3, 10, 7, 5, 12, 8, 10]).groupBy(lambda x: x % 2)

# [+] .collect()를 호출하여 결과를 드라이버로 가져온 뒤 출력
for k, v in res.collect():
    print(k, list(v))

1 [3, 7, 5]
0 [10, 12, 8, 10]


### **aggregate( ) 메서드**

In [26]:
# reduce(), fold(): 입출력 RDD 형식이 같지 않으면 오류가 발생
sc.parallelize([1, 2, 3, 4])\
    .reduce(lambda x, y: (x + y, x * y)) 
    # single value -> pair 생성 시도할 경우 오류

Py4JJavaError: An error occurred while calling z:org.apache.spark.api.python.PythonRDD.collectAndServe.
: org.apache.spark.api.python.PythonException: [PYTHON_EXCEPTION] An exception was thrown from the Python worker: Traceback (most recent call last):
  File "C:\Users\eunjin\AppData\Local\Programs\Python\Python312\Lib\site-packages\pyspark\python\lib\pyspark.zip\pyspark\worker.py", line 3622, in main
    process()
  File "C:\Users\eunjin\AppData\Local\Programs\Python\Python312\Lib\site-packages\pyspark\python\lib\pyspark.zip\pyspark\worker.py", line 3612, in process
    serializer.dump_stream(out_iter, outfile)
  File "C:\Users\eunjin\AppData\Local\Programs\Python\Python312\Lib\site-packages\pyspark\python\lib\pyspark.zip\pyspark\serializers.py", line 257, in dump_stream
    vs = list(itertools.islice(iterator, batch))
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\eunjin\AppData\Local\Programs\Python\Python312\Lib\site-packages\pyspark\core\rdd.py", line 1789, in f

In [27]:
sc.parallelize([1, 2, 3, 4]).fold((0, 0), lambda x, y: (x[0] + y, x[1] + 1))

TypeError: unsupported operand type(s) for +: 'int' and 'tuple'

In [28]:
"""
    aggregate(): 입출력 형식 다른 RDD 에 대한 reduction을 수행
    - zeroValues: 파티션 초기 값
    - seqOp: 각 파티션에 대한 reduction 함수
    - combop: 파티션 결과를 합치는 reduction 함수
""" 

seqOp = (lambda x, y: (x[0] + y, x[1] + 1))
combOp = (lambda x, y: (x[0] + y[0], x[1] + y[1]))

In [29]:
# aggregate() 수행
sc.parallelize([1, 2, 3, 4], 2).aggregate((0,0), seqOp, combOp)

(10, 4)